# **Kaggle competition -  Home Credit Risk Prediction - Unbalanced LGB Ensemble train - Experimental**

This is my first Kaggle competition. While I have worked on datasets through the MIT Professional course, this is the first real world dataset. This dataset is complex and huge, hence I plan to take a systematic step by step approach. Understanding the dataset is of utmost importance for a successful data scientist. A good insight into data will help me make better decisions aboout the aggregation I would like to make and any feature engineering once I have a hanlde of all data and features I have used to achieve best possible results. Hence, I will be taking a slow and incremental change approach. This will not only make tracing changes easier, but also enhance my learning by letting me better understand what step leads to what change.

I have learned a lot from fellow Kagglers who are gracious in sharing their code as well as their knowledge. I have extensively refered to some of the following notebooks for my learning process.

Reference files for this is:

- https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training
- https://www.kaggle.com/code/dksdms4/lb-0-565-improved-baseline-notebook
- https://www.kaggle.com/code/greysky/home-credit-baseline
- https://www.kaggle.com/code/ravi20076/homecredit-starter-inference-v1
- https://www.kaggle.com/code/peizhengwang/lb-0-57-mod-weight-pure-lgb

In [ ]:
#!pip install lightgbm

### **Version information**

- This is version 6.

In this version,
- All tables included
- dates are all changed to months except age which is in years
- Final filtering before split
- I build base model and few hyperparameter tuned models here. Other notebooks train1 and train2 do the training for the models.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing

# import further libraries
import matplotlib.pyplot as plt
import seaborn as sns

# library for cross validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

# library for metrics
from sklearn.metrics import roc_auc_score

# library for LightGBM Model
import optuna.integration.lightgbm as lgb
import lightgbm as lgbm
from lightgbm import LGBMClassifier

# library to read and write files
import pickle

# library to save and load models
import joblib

# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# library for garbage collection
import gc  # since the data is huge here, regularly cleaning up will free up memory

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator, RegressorMixin

# library to catch and ignore warnings
import warnings
warnings.filterwarnings("ignore")

# library for utility script containing various pipelines
import homecreditutility_v9 as hcu #version7

### **Loading data**

In [ ]:
# loading data into train_df
train_df = pd.read_csv("/kaggle/input/v12-data-prep-homecreditrisk2024/train_df.csv")
train_df.info()

In [ ]:
with open('/kaggle/input/v12-data-prep-homecreditrisk2024/train_cat_cols.pkl', 'rb') as f:
     cat_cols = pickle.load(f)
        
train_df[cat_cols] = train_df[cat_cols].astype('category')
train_df.info()

In [ ]:
train_df.sample(5)

### **Prepare for cross validation**

In [ ]:
y = train_df["target"]
weeks = train_df["WEEK_NUM"]
X = train_df.drop(columns=["target", "case_id", "WEEK_NUM"], axis = 1)
cv = StratifiedGroupKFold(n_splits=5, shuffle=False)

In [ ]:
X.shape, y.shape

In [ ]:
# garbage collection
hcu.MemoryOptimizer.CleanMemory()
gc.collect()

## **Function to determine gini_stability**

In [ ]:
def gini_stability(base, w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", "predict"]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", "predict"]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x["predict"])-1).tolist()

    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    print("y_hat = {}*x + {}".format(a,b))
    residuals = y - y_hat
    res_std = np.std(residuals)
    print("Residual Std. Dev:",res_std)
    avg_gini = np.mean(gini_in_time)
    print("Mean Gini in time:", avg_gini)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std

## **Build CV LightGBM hypertuned Models**

### **LightGBM hypertuned (train 1 model 1)**

In [ ]:
# baseline parameters for comparing changes done to the features (0.554)
params = {"boosting_type": "gbdt",
          "objective": "binary",
          "metric": "auc",
          "max_depth": 8,
          "max_bin": 255,
          "learning_rate": 0.05,
          "n_estimators": 2000,
          "colsample_bytree": 0.8,
          "colsample_bynode": 0.8,
        #  "class_weight": "balanced",
          "verbose": -1,
          "random_state": 42,
          "device": "gpu",}

In [ ]:
# changing categories to str type
X[cat_cols] = X[cat_cols].astype(str)

In [ ]:
%%time

fitted_models_lgb = []
cv_scores_lgb = []
oof_pred = np.zeros(X.shape[0])

for idx_train, idx_valid in cv.split(X, y, groups=weeks):#
    X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]# 
    X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]
    
    X_train[cat_cols] = X_train[cat_cols].astype("category")
    X_valid[cat_cols] = X_valid[cat_cols].astype("category")
    
    model = lgbm.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set = [(X_valid, y_valid)],
        callbacks = [lgb.log_evaluation(100), lgb.early_stopping(100)] )
    
    fitted_models_lgb.append(model)
    val_pred = model.predict_proba(X_valid)[:, 1]
    oof_pred[idx_valid] = val_pred  
    auc_score = roc_auc_score(y_valid, val_pred)
    cv_scores_lgb.append(auc_score)
    gc.collect()
    
print("CV AUC scores: ", cv_scores_lgb)
print("Maximum CV AUC score: ", max(cv_scores_lgb))
roc_auc_oof = roc_auc_score(y, oof_pred)
print("CV roc_auc_oof: ", roc_auc_oof)

In [ ]:
# save models
joblib.dump(fitted_models_lgb, 'fitted_models_lgb.joblib')
#joblib.dump(oof_pred, "oof_pred.pkl")
joblib.dump((train_df.columns, cat_cols), 'train_cat_columns.pkl')

### **Batch Predictions**

Code borrowed from:
https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training

In [ ]:
def predict_proba_in_batches(model, data, batch_size=100000):
    num_samples = len(data)
    num_batches = int(np.ceil(num_samples / batch_size))
    probabilities = np.zeros((num_samples,))

    for batch_idx in range(num_batches):
        print(f"Processing batch: {batch_idx+1}/{num_batches}")
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, num_samples)
        X_batch = data.iloc[start_idx:end_idx]
        batch_probs = model.predict_proba(X_batch)[:, 1]
        probabilities[start_idx:end_idx] = batch_probs
        gc.collect()

    return probabilities

### **Voting Model**

borrowed from https://www.kaggle.com/code/dksdms4/lb-0-565-improved-baseline-notebook

In [ ]:
class VotingModel(BaseEstimator, RegressorMixin):
    def __init__(self, estimators):
        super().__init__()
        self.estimators = estimators
        
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        y_preds = [estimator.predict(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)
    
    def predict_proba(self, X):
        y_preds = [estimator.predict_proba(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)

In [ ]:
#fitted_models = fitted_models_lgb

#vmodel1 = VotingModel(fitted_models_lgb)

In [ ]:
#%%time
#X[cat_cols] = X[cat_cols].astype("category")

# predict using the trained gbm model
#y_pred = pd.Series(predict_proba_in_batches(vmodel1, X), index=X.index)

# display AUC score for the train and validation datasets
#print(f'The AUC score on the train set is: {roc_auc_score(y, y_pred)}')

In [ ]:
#%%time
#pred_df = train_df[["WEEK_NUM", "target"]].copy()
#pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
#stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
#print(f'The stability score on the train set is: {stability_score_train}')

### **sklearn Voting Classifier**

In [ ]:
from sklearn.ensemble import VotingClassifier

oof_models_dict = [(str(i), model) for i, model in enumerate(fitted_models_lgb)]

vmodel2 = VotingClassifier(
    estimators=oof_models_dict, 
    voting = 'soft'
)

vmodel2.estimators_ = fitted_models_lgb

In [ ]:
# save model
joblib.dump(vmodel2, 'voting_classifier.joblib')

In [ ]:
%%time
X[cat_cols] = X[cat_cols].astype("category")

# predict using the trained gbm model
y_pred = pd.Series(predict_proba_in_batches(vmodel2, X), index=X.index)

# display AUC score for the train and validation datasets
print(f'The AUC score on the train set is: {roc_auc_score(y, y_pred)}')

In [ ]:
%%time
pred_df = train_df[["WEEK_NUM", "target"]].copy()
pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
print(f'The stability score on the train set is: {stability_score_train}')

In [ ]:
plt.figure(figsize = (15,10))
sns.kdeplot(x=train_df['target'], color = 'blue', label = 'target')
sns.kdeplot(x=y_pred, color = 'red', label = 'prediction')
plt.legend(loc= 'upper right')
plt.show()